# Competition: Contrail Segmentation


---

This notebook does four things and then stops: it attaches the dataset, shows you what is inside it,
plots a few records next to their masks, and writes a valid (empty) submission file.

That is on purpose. The dataset class, the dataloader, the model, the loss and the threshold are
yours to write — see the [competition README](https://github.com/tudelft/AE4353-Y26/blob/main/competition/README.md).

## Table of Contents
1. [Kaggle Setup](#kaggle-setup)
2. [What Is In The Files](#what-is-in-the-files)
3. [Looking At The Data](#looking-at-the-data)
4. [Your First Submission](#your-first-submission)
5. [Write-Up](#write-up)

## Kaggle Setup
---
**You do not need to download or upload anything.** The competition data is already published on
Kaggle — you only have to attach it to this notebook.

1. In the panel on the right, click `+ Add Input`.
2. Select the `Competitions` tab.
3. Search for `AE4353 2026` and add *AE4353 2026 - Reading Between the Lines*.

> ⚠️ **The exact mount path depends on your account.** It may be
> `/kaggle/input/competitions/ae4353-2026/`, or something else. Do not guess it: expand the data in
> the Input panel, hover over its folder, and click the copy-path icon that appears. Paste it into
> the cell below — everything else in this notebook is built from that one variable.

In [ ]:
# -------------------------------------------------------------------
#  Paste the path you copied from the Input panel on the right here:
# -------------------------------------------------------------------
DATASET_PATH = "/kaggle/input/competitions/ae4353-2026"
# -------------------------------------------------------------------

import glob
import os
import shutil

import h5py
import numpy as np
import matplotlib.pyplot as plt

if not os.path.isdir(DATASET_PATH):
    raise FileNotFoundError(
        f"There is no folder at {DATASET_PATH}.\n"
        "Copy the real path from the Input panel on the right: expand the "
        "competition data, hover over its folder, and click the copy-path icon."
    )

shutil.copy(f"{DATASET_PATH}/metric.py", "/kaggle/working/metric.py")
from metric import rle_encode, rle_decode, global_dice, write_submission

train_file = f"{DATASET_PATH}/train_000.h5"
test_files = sorted(glob.glob(f"{DATASET_PATH}/test_*.h5"))
print("Ready.")

## What Is In The Files
---
The data is stored as HDF5, sharded into a few large files. Each file holds the record ids, the
satellite imagery, and — for the training shards only — the masks you are trying to predict.

Execute the cell below to print the structure and the shapes.

In [ ]:
with h5py.File(train_file, "r") as f:
    for name, dataset in f.items():
        print(f"{name:8s} {str(dataset.shape):22s} {dataset.dtype}")

    print("\nattributes on /bands:")
    for key, value in f["bands"].attrs.items():
        print(f"  {key:8s} {value}")

    masks = f["mask"][:]
    print(f"\nrecords in this shard:     {len(masks)}")
    print(f"records with a contrail:   {np.array([m.any() for m in masks]).mean():.1%}")
    print(f"pixels that are contrail:  {masks.mean():.3%}")

That last number is the one to remember. **Around half a percent of pixels are contrail.** A model
trained with a plain binary cross-entropy loss will learn to predict nothing at all, and score zero.

The imagery is stored as integers to keep the files small. Convert it back to brightness temperature
in kelvin using the `scale` and `offset` attributes printed above.

## Looking At The Data
---
Three infrared bands, at 8.4, 11.2 and 12.3 µm. Below they are plotted as they are stored, next to
the mask.

Contrails are thin, faint and linear, and they are much easier to see in some combination of the
bands than in any single one. Finding a representation that makes them stand out is part of the
problem — and one of the cheapest places to gain. Exercise 2 asked you the same question about
polarisation channels.

In [ ]:
n_samples = 4

with h5py.File(train_file, "r") as f:
    scale, offset = f["bands"].attrs["scale"], f["bands"].attrs["offset"]
    # pick records that actually contain a contrail, so there is something to see
    search = min(200, f["bands"].shape[0])
    indices = [i for i in range(search) if f["mask"][i].any()][:n_samples]
    bands = f["bands"][indices].astype(np.float32) * scale + offset
    masks = f["mask"][indices]

fig, axes = plt.subplots(n_samples, 4, figsize=(12, 3 * n_samples))
for row, (band_stack, mask) in enumerate(zip(bands, masks)):
    for col, (image, title) in enumerate(zip(band_stack, ["8.4 um", "11.2 um", "12.3 um"])):
        axes[row, col].imshow(image, cmap="gray")
        axes[row, col].axis("off")
        if row == 0:
            axes[row, col].set_title(title, fontsize=10)

    axes[row, 3].imshow(band_stack[1], cmap="gray")
    axes[row, 3].imshow(np.ma.masked_where(mask == 0, mask), cmap="autumn", alpha=0.9)
    axes[row, 3].axis("off")
    if row == 0:
        axes[row, 3].set_title("mask", fontsize=10)

plt.suptitle("Three infrared bands and the contrail mask", fontsize=14)
plt.tight_layout()
plt.show()

## Your First Submission
---
Before training anything, make a submission that is empty. It scores exactly `0.0000`, which is a
useless model but a working pipeline — and from then on every number you see is about your model
rather than about your file format.

> ⚠️ Masks are run-length encoded, and the pixel numbering is **column-major**. Because the masks are
> square, getting that backwards produces no error — it silently submits the transpose of your mask,
> which scores well under `0.1`. Use `rle_encode()` and `write_submission()` and it cannot happen.

In [ ]:
# every test shard, not just the first -- a submission missing rows is rejected
record_ids = []
for path in test_files:
    with h5py.File(path, "r") as f:
        record_ids += [rid.decode() for rid in f["id"][:]]

empty = np.zeros((len(record_ids), 256, 256), dtype=bool)
write_submission(record_ids, empty, "/kaggle/working/submission.csv")

print(f"wrote {len(record_ids)} rows")
print(open("/kaggle/working/submission.csv").read()[:120], "...")

# the metric is the same code the leaderboard runs -- use it on your own validation split
print("\nsanity check, empty vs empty:", global_dice(empty, empty))
print("sanity check, empty vs a real mask:", global_dice(empty[:1], masks[:1]))

Download `submission.csv` from the `Output` panel on the right and upload it on the competition's
`Submit Prediction` page.

---

That is everything we give you. From here you need a `Dataset` that reads these HDF5 files, a
dataloader, a model, a loss that survives 0.5 % positive pixels, a validation split, and a threshold
chosen with `global_dice()`.

> 💡 Do not open the HDF5 file in your `Dataset.__init__` and keep the handle. With `num_workers > 0`
> the handle is duplicated into every worker, and HDF5 does not survive that — you get silently
> wrong samples rather than an error. Open it lazily, on first use, inside each worker.

## Write-Up
---
Fill this in and hand it in with this notebook. **Maximum 400 words in total.**

1. **Model.** What architecture did you use, and why that one?

2. **Loss.** What did you train with, and how did you handle the fact that ~0.5 % of pixels are
   positive?

3. **Threshold.** What threshold did you binarise at, and how did you choose it? Report your
   validation global Dice at that threshold.

4. **What didn't work.** Name one thing you tried that made the score worse, and what you think
   happened.

5. **Sources.** List every external notebook, repository, paper, model or set of pretrained weights
   you used or drew on, with links. "None" is a valid answer.